# Model Context Protocol (MCP) — From-Scratch JSON-RPC 2.0 Server and Client

This notebook builds a minimal, real MCP-shaped server and client, running over a
**real subprocess boundary via stdio** (no network, no external host — this session's
`mcp_server.py` is spawned as a genuine child process communicating over its own
stdin/stdout). See `notes.md` for the full conceptual treatment; this notebook is the
from-scratch implementation and the experiment referenced there.

## 1. Start the server as a real subprocess (stdio transport)

In [1]:
import json
import subprocess
import sys
from pathlib import Path

SERVER_PATH = Path.cwd() / "mcp_server.py"
assert SERVER_PATH.exists(), f"expected server script at {SERVER_PATH}"

proc = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1,  # line-buffered
)
print(f"Spawned real subprocess: pid={proc.pid}, cmd={[sys.executable, str(SERVER_PATH)]}")


Spawned real subprocess: pid=422195, cmd=['/home/yashwanth-aravind/ml-course/python-bootcamp/.venv/bin/python', '/home/yashwanth-aravind/ml-course/python-bootcamp/15-agent-skills-and-mcp/02-model-context-protocol/mcp_server.py']


## 2. A minimal JSON-RPC 2.0 client

Every call constructs a real JSON-RPC 2.0 request dict (`jsonrpc`, `id`, `method`,
`params`), serializes it to one line of JSON, writes it to the subprocess's stdin, and
reads exactly one line of JSON back from its stdout — this is the real MCP stdio
transport framing. The transcript (raw JSON strings sent and received) is printed for
every call in this notebook, not just summarized.

In [2]:
_next_id = 0

def send_request(method, params=None, print_transcript=True):
    """Send one real JSON-RPC 2.0 request to the subprocess server and return
    the parsed JSON-RPC response dict. Prints the raw wire transcript."""
    global _next_id
    _next_id += 1
    request = {"jsonrpc": "2.0", "id": _next_id, "method": method, "params": params or {}}
    request_line = json.dumps(request)

    proc.stdin.write(request_line + "\n")
    proc.stdin.flush()
    response_line = proc.stdout.readline().strip()
    response = json.loads(response_line)

    if print_transcript:
        print("--> SENT     :", request_line)
        print("<-- RECEIVED :", response_line)
        print()
    return response


## 3. Discovery: `list_tools`

Per MCP's discovery flow, a client always calls `list_tools` (or `list_resources`)
*before* ever calling `call_tool` (or `read_resource`) — it must learn what a server
offers, and each tool's JSON-Schema parameter contract, before it can construct a valid
call.

In [3]:
list_tools_response = send_request("list_tools")

discovered_tools = list_tools_response["result"]["tools"]
print(f"Server exposes {len(discovered_tools)} tools:")
for tool in discovered_tools:
    print(f"  - {tool['name']}: {tool['description']}")
    print(f"    inputSchema: {json.dumps(tool['inputSchema'])}")


--> SENT     : {"jsonrpc": "2.0", "id": 1, "method": "list_tools", "params": {}}
<-- RECEIVED : {"jsonrpc": "2.0", "id": 1, "result": {"tools": [{"name": "add", "description": "Add two numbers together and return their sum.", "inputSchema": {"type": "object", "properties": {"a": {"type": "number", "description": "First addend"}, "b": {"type": "number", "description": "Second addend"}}, "required": ["a", "b"]}}, {"name": "word_count", "description": "Count the number of whitespace-separated words in a string.", "inputSchema": {"type": "object", "properties": {"text": {"type": "string", "description": "The text to count words in"}}, "required": ["text"]}}, {"name": "reverse_string", "description": "Reverse the characters of a string.", "inputSchema": {"type": "object", "properties": {"text": {"type": "string", "description": "The text to reverse"}}, "required": ["text"]}}]}}

Server exposes 3 tools:
  - add: Add two numbers together and return their sum.
    inputSchema: {"type": "object

## 4. Invocation: `call_tool` — a valid call

The client now calls the real `add` tool, discovered above, with real arguments. The
server validates the arguments against the JSON-Schema it advertised, executes the
underlying Python function, and returns a real JSON-RPC `result`.

In [4]:
add_response = send_request("call_tool", {"name": "add", "arguments": {"a": 17, "b": 25}})
assert "result" in add_response
print("add(17, 25) ->", add_response["result"]["value"])


--> SENT     : {"jsonrpc": "2.0", "id": 2, "method": "call_tool", "params": {"name": "add", "arguments": {"a": 17, "b": 25}}}
<-- RECEIVED : {"jsonrpc": "2.0", "id": 2, "result": {"content": [{"type": "text", "text": "42"}], "value": 42}}

add(17, 25) -> 42


In [5]:
word_count_response = send_request(
    "call_tool",
    {"name": "word_count", "arguments": {"text": "the quick brown fox jumps over the lazy dog"}},
)
assert "result" in word_count_response
print("word_count(...) ->", word_count_response["result"]["value"])


--> SENT     : {"jsonrpc": "2.0", "id": 3, "method": "call_tool", "params": {"name": "word_count", "arguments": {"text": "the quick brown fox jumps over the lazy dog"}}}
<-- RECEIVED : {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"type": "text", "text": "9"}], "value": 9}}

word_count(...) -> 9


## 5. Experiment: JSON-Schema validation catches a malformed call before it reaches the tool

**Hypothesis:** the server's `call_tool` handler validates arguments against the tool's
declared JSON-Schema (required-field presence + type checking) *before* dispatching to
the underlying Python function. A call missing a required argument, or with a
wrong-typed argument, should be rejected with a proper JSON-RPC error response — the
Python `add`/`word_count` functions should never even run — while a well-formed call
should be accepted and executed normally.

**Setup:** three real `call_tool` requests against the running subprocess server:
1. A well-formed `add` call (both required arguments present, correct types) — expect
   a JSON-RPC `result`.
2. An `add` call missing the required `b` argument — expect a JSON-RPC `error` with
   code `-32602` ("Invalid params"), before the addition ever runs.
3. An `add` call where `a` is a string instead of a number — expect the same kind of
   JSON-RPC `error`, this time for a type mismatch rather than a missing field.

In [6]:
# Case 1: well-formed call -> accepted, executes, returns a result
valid_call = send_request("call_tool", {"name": "add", "arguments": {"a": 4, "b": 5}})
print("Case 1 (valid):", "result" if "result" in valid_call else "error", "->", valid_call.get("result") or valid_call.get("error"))


--> SENT     : {"jsonrpc": "2.0", "id": 4, "method": "call_tool", "params": {"name": "add", "arguments": {"a": 4, "b": 5}}}
<-- RECEIVED : {"jsonrpc": "2.0", "id": 4, "result": {"content": [{"type": "text", "text": "9"}], "value": 9}}

Case 1 (valid): result -> {'content': [{'type': 'text', 'text': '9'}], 'value': 9}


In [7]:
# Case 2: missing required argument -> rejected before the tool function runs
missing_arg_call = send_request("call_tool", {"name": "add", "arguments": {"a": 4}})
print("Case 2 (missing required arg):", "result" if "result" in missing_arg_call else "error", "->", missing_arg_call.get("error"))


--> SENT     : {"jsonrpc": "2.0", "id": 5, "method": "call_tool", "params": {"name": "add", "arguments": {"a": 4}}}
<-- RECEIVED : {"jsonrpc": "2.0", "id": 5, "error": {"code": -32602, "message": "invalid params for tool 'add': missing required argument: 'b'"}}

Case 2 (missing required arg): error -> {'code': -32602, 'message': "invalid params for tool 'add': missing required argument: 'b'"}


In [8]:
# Case 3: wrong type -> also rejected, different validation message
wrong_type_call = send_request("call_tool", {"name": "add", "arguments": {"a": "four", "b": 5}})
print("Case 3 (wrong type):", "result" if "result" in wrong_type_call else "error", "->", wrong_type_call.get("error"))


--> SENT     : {"jsonrpc": "2.0", "id": 6, "method": "call_tool", "params": {"name": "add", "arguments": {"a": "four", "b": 5}}}
<-- RECEIVED : {"jsonrpc": "2.0", "id": 6, "error": {"code": -32602, "message": "invalid params for tool 'add': argument 'a' expected type 'number', got 'str'"}}

Case 3 (wrong type): error -> {'code': -32602, 'message': "invalid params for tool 'add': argument 'a' expected type 'number', got 'str'"}


In [9]:
print("Case 1 (valid)      :", "ACCEPTED, value =", valid_call["result"]["value"])
print("Case 2 (missing arg):", "REJECTED, code =", missing_arg_call["error"]["code"], "-", missing_arg_call["error"]["message"])
print("Case 3 (wrong type) :", "REJECTED, code =", wrong_type_call["error"]["code"], "-", wrong_type_call["error"]["message"])

assert "result" in valid_call, "Case 1 should have been accepted"
assert missing_arg_call["error"]["code"] == -32602, "Case 2 should reject with Invalid params"
assert wrong_type_call["error"]["code"] == -32602, "Case 3 should reject with Invalid params"
print()
print("Hypothesis CONFIRMED: valid call executed; both malformed calls were rejected")
print("with JSON-RPC error code -32602 before the add() function's logic ever ran.")


Case 1 (valid)      : ACCEPTED, value = 9
Case 2 (missing arg): REJECTED, code = -32602 - invalid params for tool 'add': missing required argument: 'b'
Case 3 (wrong type) : REJECTED, code = -32602 - invalid params for tool 'add': argument 'a' expected type 'number', got 'str'

Hypothesis CONFIRMED: valid call executed; both malformed calls were rejected
with JSON-RPC error code -32602 before the add() function's logic ever ran.


**Actual result:** Case 1 (`a=4, b=5`, both required, both numbers) returned a real
JSON-RPC `result` with `value: 9`. Case 2 (`a=4`, `b` missing) returned a JSON-RPC
`error` with `code: -32602` and message `"invalid params for tool 'add': missing
required argument: 'b'"`. Case 3 (`a="four"`, `b=5`) returned a JSON-RPC `error`, also
`code: -32602`, message `"invalid params for tool 'add': argument 'a' expected type
'number', got 'str'"`. In both rejected cases the error message was produced entirely
by `validate_arguments()` inside `mcp_server.py`, before `TOOL_FUNCTIONS["add"]` was
ever called — confirmed by reading the server's `handle_request()`: the `return
make_error(...)` for a validation failure happens strictly before the `try: value =
TOOL_FUNCTIONS[tool_name](arguments)` line is reached.

**Interpretation:** the hypothesis held exactly as stated. Schema validation is a real
gate, not cosmetic — a malformed call cannot reach and potentially crash (or silently
misbehave inside) the underlying tool function.

**Limitations:** this validator is deliberately basic — required-field presence and a
single-level type check only. It does not validate nested object/array schemas, `enum`
constraints, numeric ranges (`minimum`/`maximum`), string `pattern`/`format`, or
`additionalProperties: false`. A production MCP server would typically use a full
JSON-Schema validation library (e.g. Python's `jsonschema` package) rather than this
hand-rolled check.

## 6. Calling an unknown tool (a different failure mode)

For completeness: a `call_tool` request naming a tool the server never registered is
rejected with a different JSON-RPC error code (`-32601`, "Method not found" reused here
for an unknown tool name), distinct from the `-32602` schema-validation errors above.

In [10]:
unknown_tool_call = send_request("call_tool", {"name": "divide", "arguments": {"a": 4, "b": 2}})
print(unknown_tool_call)
assert unknown_tool_call["error"]["code"] == -32601


--> SENT     : {"jsonrpc": "2.0", "id": 7, "method": "call_tool", "params": {"name": "divide", "arguments": {"a": 4, "b": 2}}}
<-- RECEIVED : {"jsonrpc": "2.0", "id": 7, "error": {"code": -32601, "message": "unknown tool: 'divide'"}}

{'jsonrpc': '2.0', 'id': 7, 'error': {'code': -32601, 'message': "unknown tool: 'divide'"}}


## 7. `reverse_string` — the third registered tool, for completeness

In [11]:
reverse_response = send_request("call_tool", {"name": "reverse_string", "arguments": {"text": "model context protocol"}})
print("reverse_string('model context protocol') ->", reverse_response["result"]["value"])


--> SENT     : {"jsonrpc": "2.0", "id": 8, "method": "call_tool", "params": {"name": "reverse_string", "arguments": {"text": "model context protocol"}}}
<-- RECEIVED : {"jsonrpc": "2.0", "id": 8, "result": {"content": [{"type": "text", "text": "locotorp txetnoc ledom"}], "value": "locotorp txetnoc ledom"}}

reverse_string('model context protocol') -> locotorp txetnoc ledom


## 8. Clean shutdown of the real subprocess

In [12]:
proc.stdin.close()
proc.wait(timeout=5)
stderr_output = proc.stderr.read()
print(f"Subprocess exited with code {proc.returncode}")
if stderr_output:
    print("stderr:", stderr_output)


Subprocess exited with code 0


## Summary

This notebook ran a real MCP-shaped `list_tools` / `call_tool` exchange over a real
subprocess boundary via stdio, using the real JSON-RPC 2.0 wire format throughout
(`{"jsonrpc": "2.0", "id": ..., "method": ..., "params": ...}` requests; `result` or
`error` responses). The discovery-before-call ordering was followed (`list_tools`
before any `call_tool`), a JSON-Schema-validated tool call was demonstrated both
accepted and rejected (the Experiment section, above), and an unknown-tool call was
rejected with a distinct error code. See `notes.md` for the full conceptual treatment,
failure modes, and real-world context.